# Working with image rotations

So far, we have only been perturbing/learning the image translations. We still need to incorporate rotations. We will build that up here.

In [1]:
from pycolmap import Image
%load_ext autoreload
%autoreload 2

import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)
# Choose device (cuda if available, else cpu)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)

2.9.0+cu128
Selected device: cuda


# Load Data

In [2]:
## Load the colmap data
import pycolmap
import pathlib
import utils.subset
import random
import utils.loaders

# Load the Colmap data into a torch dataset
model_path = pathlib.Path('./output/01_nerf/02_lego_large/1762043798762/0')
reconstruction = pycolmap.Reconstruction(model_path)

# Display rotations

In [3]:
from pycolmap import Image
example_img: Image = next(iter(reconstruction.images.values()))
example_img

Image(image_id=57, camera=Camera(camera_id=1), name="r_57.png", has_pose=1, triangulated=677/3683)

In [4]:
help(example_img)

Help on Image in module pycolmap._core object:

class Image(pybind11_builtins.pybind11_object)
 |  Method resolution order:
 |      Image
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __copy__(...)
 |      __copy__(self: pycolmap._core.Image) -> pycolmap._core.Image
 |
 |  __deepcopy__(...)
 |      __deepcopy__(self: pycolmap._core.Image, arg0: dict) -> pycolmap._core.Image
 |
 |  __getstate__(...)
 |      __getstate__(self: pycolmap._core.Image, /) -> dict
 |
 |  __init__(...)
 |      __init__(*args, **kwargs)
 |      Overloaded function.
 |
 |      1. __init__(self: pycolmap._core.Image) -> None
 |
 |      2. __init__(self: pycolmap._core.Image, name: str = '', points2D: pycolmap._core.Point2DList = Point2DList(), camera_id: typing.SupportsInt = pycolmap.INVALID_CAMERA_ID, image_id: typing.SupportsInt = pycolmap.INVALID_IMAGE_ID) -> None
 |
 |      3. __init__(self: pycolmap._core.Image, name: str = '', keypoints: typing.Annotat

In [5]:
example_img.projection_center()

array([-1.42097578,  1.09834433, -2.01780405])

In [6]:
example_img.viewing_direction()

array([ 0.09279061, -0.02899918,  0.99526326])

In [7]:
example_img.cam_from_world()

Rigid3d(rotation_xyzw=[-0.0200878, -0.0443267, -0.122134, 0.99132], translation=[0.93765, -1.30404, 2.17195])

In [8]:
help(example_img.cam_from_world())

Help on Rigid3d in module pycolmap._core object:

class Rigid3d(pybind11_builtins.pybind11_object)
 |  Method resolution order:
 |      Rigid3d
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __copy__(...)
 |      __copy__(self: pycolmap._core.Rigid3d) -> pycolmap._core.Rigid3d
 |
 |  __deepcopy__(...)
 |      __deepcopy__(self: pycolmap._core.Rigid3d, arg0: dict) -> pycolmap._core.Rigid3d
 |
 |  __getstate__(...)
 |      __getstate__(self: pycolmap._core.Rigid3d, /) -> dict
 |
 |  __init__(...)
 |      __init__(*args, **kwargs)
 |      Overloaded function.
 |
 |      1. __init__(self: pycolmap._core.Rigid3d) -> None
 |
 |      2. __init__(self: pycolmap._core.Rigid3d, rotation: pycolmap._core.Rotation3d, translation: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[3, 1]"]) -> None
 |
 |      3. __init__(self: pycolmap._core.Rigid3d, matrix: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[3, 4]"]) -> None
 |
 |  

In [9]:
help(example_img.cam_from_world().rotation)

Help on Rotation3d in module pycolmap._core object:

class Rotation3d(pybind11_builtins.pybind11_object)
 |  Method resolution order:
 |      Rotation3d
 |      pybind11_builtins.pybind11_object
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __copy__(...)
 |      __copy__(self: pycolmap._core.Rotation3d) -> pycolmap._core.Rotation3d
 |
 |  __deepcopy__(...)
 |      __deepcopy__(self: pycolmap._core.Rotation3d, arg0: dict) -> pycolmap._core.Rotation3d
 |
 |  __getstate__(...)
 |      __getstate__(self: pycolmap._core.Rotation3d, /) -> dict
 |
 |  __init__(...)
 |      __init__(*args, **kwargs)
 |      Overloaded function.
 |
 |      1. __init__(self: pycolmap._core.Rotation3d) -> None
 |
 |      2. __init__(self: pycolmap._core.Rotation3d, xyzw: typing.Annotated[numpy.typing.ArrayLike, numpy.float64, "[4, 1]"]) -> None
 |
 |      Quaternion in [x,y,z,w] format.
 |
 |      3. __init__(self: pycolmap._core.Rotation3d, matrix: typing.Annotated[numpy.typing.ArrayLike, numpy.flo

In [10]:
example_img.cam_from_world().rotation.quat

array([-0.02008776, -0.04432667, -0.12213427,  0.99131975])

In [11]:
# Test the new plot_rigid3d_poses function
from utils.vis.plot_colmap import plot_rigid3d_poses

# Get the cam_from_world transformation for the example image
cam_from_world = example_img.cam_from_world()

# Plot it
fig = plot_rigid3d_poses(cam_from_world, labels="Example Camera", show_origin=True, axis_length=0.2)
fig.show()

In [12]:
# Plot multiple camera poses
# Get a few camera poses from the reconstruction
camera_poses = []
camera_labels = []
for i, img in enumerate(list(reconstruction.images.values())[:]):
    word_from_cam = img.cam_from_world().inverse()
    camera_poses.append(word_from_cam)
    camera_labels.append(img.name)

fig = plot_rigid3d_poses(camera_poses, labels=camera_labels, show_origin=True, axis_length=0.3)
fig.show()

# Rotate a single image pose

Let's try rotating a single image pose.

In [13]:
# This is the transformation that will take a pose in world space, and transform the pose to the camera pose space.
cam_from_world = example_img.cam_from_world()
cam_from_world

Rigid3d(rotation_xyzw=[-0.0200878, -0.0443267, -0.122134, 0.99132], translation=[0.93765, -1.30404, 2.17195])

In [14]:
# When we take the inverse, this gives us the camera pose relative to the world space.
world_from_cam = cam_from_world.inverse()
world_from_cam

Rigid3d(rotation_xyzw=[0.0200878, 0.0443267, 0.122134, 0.99132], translation=[-1.42098, 1.09834, -2.0178])

In [15]:
# Notice that the above translation is the same as our projection_center!
example_img.projection_center()

array([-1.42097578,  1.09834433, -2.01780405])

In [16]:
import numpy as np
from scipy.spatial.transform import Rotation

def get_rotation_quat(x, y, z):
    # Scipy gives xyzw, rolling by 1 shifts w from the end to the front
    return Rotation.from_euler('xyz', [x, y, z], degrees=True).as_quat()

In [33]:
# Or, we could use the built-in random quat generator from scipy
Rotation.random().as_quat()

array([ 0.44825701,  0.11902623, -0.82806499,  0.31497108])

In [17]:
# Define your rotations
orbit_quat = get_rotation_quat(1, 0, 0)
orbit_quat

array([0.00872654, 0.        , 0.        , 0.99996192])

In [18]:
# Create transform (pivoting around 0,0,0)
orbit_transform = pycolmap.Rigid3d(orbit_quat, np.zeros(3))
orbit_transform

Rigid3d(rotation_xyzw=[0.00872654, 0, 0, 0.999962], translation=[0, 0, 0])

Let's test mutability/reference properties to make sure we understand

In [19]:
a = orbit_transform
a

Rigid3d(rotation_xyzw=[0.00872654, 0, 0, 0.999962], translation=[0, 0, 0])

In [20]:
a.translation = a.translation + 1
a

Rigid3d(rotation_xyzw=[0.00872654, 0, 0, 0.999962], translation=[1, 1, 1])

In [170]:
b = orbit_transform.__copy__()

In [171]:
b

Rigid3d(rotation_xyzw=[0.00872654, 0, 0, 0.999962], translation=[1, 1, 1])

In [173]:
b.translation = b.translation + 1
b

Rigid3d(rotation_xyzw=[0.00872654, 0, 0, 0.999962], translation=[2, 2, 2])

In [28]:
eg_1 = pycolmap.Rigid3d(get_rotation_quat(180, 0, 0), np.zeros(3))
eg_1

Rigid3d(rotation_xyzw=[1, 0, 0, 6.12303e-17], translation=[0, 0, 0])

In [31]:
# if we mutate the inverse, does this mutate the original Rigid3d?
eg_2 = eg_1.inverse()
print(eg_2)
eg_2.translation = eg_2.translation + 1
print(eg_2)
print(eg_1)

Rigid3d(rotation_xyzw=[-1, -0, -0, 6.12303e-17], translation=[0, 0, 0])
Rigid3d(rotation_xyzw=[-1, -0, -0, 6.12303e-17], translation=[1, 1, 1])
Rigid3d(rotation_xyzw=[1, 0, 0, 6.12303e-17], translation=[0, 0, 0])


Nope! Good stuff. the original is not mutated.

In [174]:
orbit_transform

Rigid3d(rotation_xyzw=[0.00872654, 0, 0, 0.999962], translation=[1, 1, 1])

In [107]:
new_world_from_cam = orbit_transform * world_from_cam
new_world_from_cam

Rigid3d(rotation_xyzw=[0.0287378, 0.0432592, 0.122516, 0.991107], translation=[-1.42098, 1.13339, -1.99833])

In [108]:
new_cam_from_world = new_world_from_cam.inverse()
new_cam_from_world

Rigid3d(rotation_xyzw=[-0.0287378, -0.0432592, -0.122516, 0.991107], translation=[0.93765, -1.30404, 2.17195])

In [109]:
# We will plot these transforms
print(world_from_cam)
print(new_world_from_cam)

Rigid3d(rotation_xyzw=[0.0200878, 0.0443267, 0.122134, 0.99132], translation=[-1.42098, 1.09834, -2.0178])
Rigid3d(rotation_xyzw=[0.0287378, 0.0432592, 0.122516, 0.991107], translation=[-1.42098, 1.13339, -1.99833])


In [110]:
# plot to see effect/
fig = plot_rigid3d_poses([world_from_cam, new_world_from_cam], labels=["old", "new"], show_origin=True, axis_length=0.3)
fig.show()

# Sanity Check
Let's plot lots to make sure the pattern looks correct

In [135]:
# Crea
cam_from_world = example_img.cam_from_world()
world_from_cam = cam_from_world.inverse()

many_rotations_around_x_axis = []
for i in range(180):
    rot_quat = get_rotation_quat(i*10, 0, 0)
    orbit_transform = pycolmap.Rigid3d(rot_quat, np.zeros(3))
    new_pose = orbit_transform * world_from_cam
    many_rotations_around_x_axis.append(new_pose)

many_rotations_around_y_axis = []
for i in range(180):
    rot_quat = get_rotation_quat(0, i*10, 0)
    orbit_transform = pycolmap.Rigid3d(rot_quat, np.zeros(3))
    new_pose = orbit_transform * world_from_cam
    many_rotations_around_y_axis.append(new_pose)

many_rotations_around_z_axis = []
for i in range(180):
    rot_quat = get_rotation_quat(0, 0, i*10)
    orbit_transform = pycolmap.Rigid3d(rot_quat, np.zeros(3))
    new_pose = orbit_transform * world_from_cam
    many_rotations_around_z_axis.append(new_pose)

In [134]:
# plot to see
fig = plot_rigid3d_poses(many_rotations_around_x_axis, show_origin=True, axis_length=0.3)
fig.show()

In [136]:
# plot to see
fig = plot_rigid3d_poses(many_rotations_around_y_axis, show_origin=True, axis_length=0.3)
fig.show()

In [137]:
# plot to see
fig = plot_rigid3d_poses(many_rotations_around_z_axis, show_origin=True, axis_length=0.3)
fig.show()

# PLot them all!

In [1]:
# plot to see
fig = plot_rigid3d_poses(many_rotations_around_x_axis + many_rotations_around_y_axis + many_rotations_around_z_axis, show_origin=True, axis_length=0.3)
fig.show()

NameError: name 'plot_rigid3d_poses' is not defined

# Rotate a collection of image poses

In [139]:
# Create a random subset using distance based heuristic
graph = utils.subset.reconstruction_to_pyg_data(reconstruction)

In [141]:
graph.x

tensor([[-1.4210e+00,  1.0983e+00, -2.0178e+00],
        [-5.0561e-01,  1.4473e+00, -1.9871e+00],
        [-8.7134e-01,  1.0277e+00, -2.0362e+00],
        [-1.4425e+00,  1.2606e-01, -1.9379e+00],
        [-1.2084e+00,  4.3559e+00, -5.2545e-01],
        [-7.2209e-01, -1.1619e+00, -1.4978e+00],
        [ 3.3094e+00,  4.6423e-02,  1.3857e+00],
        [-1.6856e+00,  1.8505e+00, -1.8977e+00],
        [-1.1060e+00,  2.3858e+00, -1.8104e+00],
        [-6.8548e-01,  2.8489e+00, -1.6200e+00],
        [-1.9203e+00, -4.3835e-01, -1.7161e+00],
        [-1.9794e+00, -8.2072e-01, -1.5525e+00],
        [-2.8840e+00, -3.1359e-01, -1.4222e+00],
        [-2.1811e-01, -3.6868e-01, -1.7666e+00],
        [ 4.0640e-01, -3.8644e-01, -1.5990e+00],
        [ 5.5233e-01, -4.7632e-01, -1.5144e+00],
        [ 2.9078e+00,  1.2795e+00,  2.0859e-01],
        [ 1.2949e+00, -3.7219e-01, -1.1836e+00],
        [ 8.5093e-01,  1.3613e+00, -1.6269e+00],
        [ 1.6326e+00,  5.3027e-01, -1.1720e+00],
        [ 1.2489e+00

In [144]:
model_path = pathlib.Path('./output/01_nerf/02_lego_large/1762043798762/0')
reconstruction1 = pycolmap.Reconstruction(model_path)
reconstruction2 = pycolmap.Reconstruction(model_path)

In [161]:
from utils.loaders.dataset import DataSfm

perturb_rot = get_rotation_quat(5, 180, 30)
perturb_translate = [1, -2, 2]
perturb_transform = pycolmap.Rigid3d(perturb_rot, perturb_translate)
perturb_transform

Rigid3d(rotation_xyzw=[-0.258573, 0.965006, -0.0421331, 0.0112895], translation=[1, -2, 2])

In [162]:
original_cams = []
new_cams = []
for img in reconstruction2.images.values():
    # Get transform in world space
    world_from_cam =  img.cam_from_world().inverse()
    original_cams.append(world_from_cam)
    # Perturb the transform
    new_world_from_cam = perturb_transform * world_from_cam
    new_cams.append(new_world_from_cam)

In [163]:
# plot to see effect/
fig = plot_rigid3d_poses(original_cams + new_cams, show_origin=True, axis_length=0.3)
fig.show()